# SageMaker Workflow :: Stakeholder Question 3

"**Predict the likely complaint category for an incoming 311 complaint.**"

Use this notebook to review and run a complete SageMaker **Linear Learner** workflow for this stakeholder question. This notebook is designed for the Day 28 in-class compare-and-contrast activity so that you can connect the SageMaker built-in algorithm workflow to the sklearn work you completed previously.

At a high level, this notebook will help you:
- Load a modeling dataset from S3,
- Preprocess the data into the format Linear Learner expects,
- Upload train and test files back to S3,
- Launch a SageMaker training job,
- Use Batch Transform to generate predictions, and
- Evaluate the model output in the notebook.

# Setup

This notebook uses SageMaker's built-in **Linear Learner** algorithm to train a baseline model for this stakeholder question. The overall workflow is different from sklearn: you will still prepare data in the notebook, but the actual training job runs on separately managed compute resources in SageMaker and reads its input data from S3.

In the cells below, you will:
- Import the libraries needed for SageMaker, pandas, and evaluation,
- Connect to your SageMaker session and execution role,
- Define your S3 bucket and folder paths, and
- Retrieve the correct regional container image for Linear Learner.

In [1]:
import io
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
s3client = boto3.client('s3')

print(f"Region: {region}")
print(f"Role: {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region: us-east-1
Role: arn:aws:iam::964165949460:role/LabRole


## S3 configuration

Update the bucket and prefix below so they match the location of your exported modeling CSV. Keeping a clear folder structure in S3 matters because the training data, test data, model artifacts, and prediction outputs are all written to separate locations rather than staying inside notebook memory as they would in sklearn.

In [3]:
# ---------------------------------------------------------------
# Update these to match your S3 bucket and folder structure
# ---------------------------------------------------------------
S3_BUCKET = 'cmse492-cranda98-nyc311-964165949460-us-east-1-an'
S3_PREFIX = 'modeling/modeling_data.csv'

LL_IMAGE = sagemaker.image_uris.retrieve('linear-learner', region)
print('worked')

worked


## Helper functions

Linear Learner expects a very specific CSV format: all features must be numeric, the label must be in the first column, and there should be no header row. These helper functions make it easier to upload properly formatted files to S3 and reload outputs such as batch prediction results.

In [4]:
def upload_csv_to_s3(df, bucket, key):
    """Upload a DataFrame as a headerless CSV to S3."""
    buf = io.BytesIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)
    s3client.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f"Uploaded s3://{bucket}/{key} with {len(df):,} rows")
    return f"s3://{bucket}/{key}"

def download_csv_from_s3(bucket, key, **kwargs):
    """Download a CSV from S3 into a DataFrame."""
    obj = s3client.get_object(Bucket=bucket, Key=key)
    data = obj['Body'].read().decode('utf-8')
    return pd.read_csv(io.StringIO(data), **kwargs)

def make_ll_dataframe(X, y):
    """
    Combine label (y) and features (X) into the format Linear Learner expects:
    label first, all numeric, no header.
    """
    if hasattr(X, 'toarray'):
        X = X.toarray()
    label_col = pd.Series(y, name='label').reset_index(drop=True)
    feat_df = pd.DataFrame(X).reset_index(drop=True)
    return pd.concat([label_col, feat_df], axis=1)

## Load and preprocess the data

This stakeholder question is a **multi-class classification** problem. The target column is `complaint_category`, which contains multiple category labels rather than a simple yes/no outcome. Before training, those category names need to be encoded as integers because Linear Learner requires class labels starting at 0.

The feature columns must also be numeric. That means categorical predictors such as borough and agency are one-hot encoded before the training and test files are uploaded to S3.

In [6]:
# Update your path to your modeling data for this stakeholder question as needed
df = download_csv_from_s3(S3_BUCKET, "modeling/modeling_data.csv")
print(df.shape)
display(df.head())

# Features and target
cat_cols = ['borough', 'agency']
num_cols = ['day_of_week', 'hour_of_day']
target = 'complaint_category'

X = df[cat_cols + num_cols]
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[target]).astype(float)

print('Class mapping:')
print(dict(enumerate(label_encoder.classes_)))

# One-hot encode categoricals; pass numerics through
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print('Train:', X_train.shape, 'Test:', X_test.shape)

(157244, 6)


,borough,incident_zip,agency,day_of_week,hour_of_day,complaint_category
0,QUEENS,11368.0,NYPD,5,21,traffic
1,BRONX,10461.0,NYPD,5,21,traffic
2,BROOKLYN,11208.0,NYPD,5,21,noise
3,BROOKLYN,11228.0,NYPD,5,21,traffic
4,MANHATTAN,10002.0,DOT,5,21,traffic


Class mapping:
{0: 'housing', 1: 'noise', 2: 'sanitation', 3: 'traffic'}
Train: (125795, 12) Test: (31449, 12)


## Upload train and test files to S3

Even though you are working inside a notebook, SageMaker's managed training job cannot directly access variables sitting in notebook memory. That is why we convert the encoded arrays into headerless CSV files, place the label first, and upload the train and test data to S3 before training begins.

In [7]:
train_path = upload_csv_to_s3(
    make_ll_dataframe(X_train, y_train),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-multiclass/train/train.csv",
)

test_path = upload_csv_to_s3(
    make_ll_dataframe(X_test, y_test),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-multiclass/test/test.csv",
)

Uploaded s3://cmse492-cranda98-nyc311-964165949460-us-east-1-an/modeling/modeling_data.csv/ll-multiclass/train/train.csv with 125,795 rows
Uploaded s3://cmse492-cranda98-nyc311-964165949460-us-east-1-an/modeling/modeling_data.csv/ll-multiclass/test/test.csv with 31,449 rows


## Train the Linear Learner model

For multiclass classification, SageMaker needs two key pieces of information: `predictor_type='multiclass_classifier'` and the total number of classes.

This is one of the clearest examples of how SageMaker built-in algorithms make you describe the job configuration explicitly rather than relying on a scikit-learn model object to infer everything locally.

In [8]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

ll_model = Estimator(
    image_uri=LL_IMAGE,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-multiclass/output",
    sagemaker_session=session,
)

ll_model.set_hyperparameters(
    predictor_type='multiclass_classifier',
    num_classes=int(len(label_encoder.classes_)),
    feature_dim=X_train.shape[1],
    mini_batch_size=1000,
    epochs=10,
    normalize_data=True,
    normalize_label=False,
)

ll_model.fit(
    {'train': TrainingInput(train_path, content_type='text/csv')},
    logs=False, # Change this to True if you'd like to see the training output.
)

INFO:sagemaker:Creating training-job with name: linear-learner-2026-04-08-13-45-52-721



2026-04-08 13:45:57 Starting - Starting the training job..
2026-04-08 13:46:12 Starting - Preparing the instances for training....
2026-04-08 13:46:34 Downloading - Downloading input data.......
2026-04-08 13:47:15 Downloading - Downloading the training image...............
2026-04-08 13:48:36 Training - Training image download completed. Training in progress..........
2026-04-08 13:49:26 Uploading - Uploading generated training model..
2026-04-08 13:49:44 Completed - Training job completed


## Run Batch Transform for predictions

Instead of deploying a real-time endpoint, this notebook uses **Batch Transform** to score the full test set. That is a good fit for your experimentation because SageMaker spins up temporary inference compute, writes predictions back to S3, and then shuts the job down when it is finished. This avoids leaving a persistent endpoint running, which would consume resources and create additional costs.

In [ ]:
transformer = ll_model.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-multiclass/predictions",
    accept='text/csv',
    assemble_with='Line',
)

# Run on the test set (features only — no label column)
test_features_path = upload_csv_to_s3(
    pd.DataFrame(X_test),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-multiclass/test/test-features.csv",
)

transformer.transform(
    test_features_path,
    content_type='text/csv',
    split_type='Line',
    wait=True,
)

print('Batch transform complete.')

## Evaluate the predictions

Because the labels were integer-encoded before training, the final step decodes the predicted class numbers back into readable category names. The notebook then reports accuracy, macro F1, and a classification report so you can compare performance across all categories instead of focusing only on the most common one.

In [ ]:
pred_key = f"{S3_PREFIX}/ll-multiclass/predictions/test-features.csv.out"
preds = download_csv_from_s3(S3_BUCKET, pred_key, header=None)
display(preds.head())

y_pred = preds.iloc[:, 0].astype(int)

y_test_names = label_encoder.inverse_transform(y_test.astype(int))
y_pred_names = label_encoder.inverse_transform(y_pred)

print('=== Multi-class Classifier Results ===')
print(f'Accuracy : {accuracy_score(y_test_names, y_pred_names):.4f}')
print(f'Macro F1 : {f1_score(y_test_names, y_pred_names, average="macro"):.4f}')
print()
print(classification_report(y_test_names, y_pred_names))

## Wrap-up

After you run the notebook, compare the results here to the sklearn workflow you used previously. As you review the outputs, pay attention to what changed conceptually: the model training happened as a SageMaker job, the data had to be staged in S3, and Batch Transform handled prediction without creating a persistent endpoint.

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.